# Chinese Understanding Benchmark v7

Compare Qwen 4B Instruct, Gemma E2B-it, and quantized Gemma E4B-it on CLUE tasks.

In [1]:

#!pip install -U transformers datasets accelerate peft trl scikit-learn pandas tqdm sentencepiece mlx-vlm mlx-lm huggingface_hub[hf_xet]


zsh:1: no matches found: huggingface_hub[hf_xet]


In [3]:
!pip install -U \
transformers \
datasets \
accelerate \
peft \
trl \
scikit-learn \
pandas \
tqdm \
sentencepiece \
mlx-lm \
mlx-vlm

In [4]:

MODELS = {
    "qwen3_4b_instruct_2507": {
        "model_id": "Qwen/Qwen3-4B-Instruct-2507",
        "backend": "transformers",
    },
    "gemma_e2b_it": {
        "model_id": "google/gemma-4-E2B-it",
        "backend": "transformers",
    },
    "gemma_e4b_it_4bit": {
        "model_id": "mlx-community/gemma-4-e4b-it-OptiQ-4bit",
        "backend": "mlx",
    },
}

TASKS = ["afqmc", "tnews", "cmnli"]
MAX_SAMPLES = 200


In [5]:

import gc
import re
import time
import pandas as pd
import torch

from tqdm import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score

from transformers import AutoTokenizer, AutoModelForCausalLM

LABEL_ALIASES = {
    "afqmc": {
        "相同": ["相同", "一致", "yes", "1"],
        "不同": ["不同", "no", "0"],
    },
    "cmnli": {
        "蕴含": ["蕴含", "entailment", "0"],
        "中立": ["中立", "neutral", "1"],
        "矛盾": ["矛盾", "contradiction", "2"],
    },
}

def normalize_output(text):
    text = str(text).strip().lower()
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text

def cleanup():
    gc.collect()
    try:
        torch.mps.empty_cache()
    except:
        pass

def build_prompt(task, ex):
    if task == "afqmc":
        return f"""你是中文二分类器。只输出“相同”或“不同”。

句子1：{ex['sentence1']}
句子2：{ex['sentence2']}

答案："""

    if task == "cmnli":
        return f"""你是中文自然语言推理分类器。只输出“蕴含”、“中立”或“矛盾”。

前提：{ex['sentence1']}
假设：{ex['sentence2']}

答案："""

    if task == "tnews":
        return f"""你是中文新闻分类器。只输出类别名称。

标题：{ex['sentence']}

答案："""

def extract_label(task, text, tnews_mapping=None):
    text = normalize_output(text)

    if task == "tnews":
        for k, v in tnews_mapping.items():
            if normalize_output(v) in text:
                return str(k)
        return "__invalid__"

    for canonical, aliases in LABEL_ALIASES[task].items():
        for alias in aliases:
            if normalize_output(alias) in text:
                if task == "afqmc":
                    return "1" if canonical == "相同" else "0"
                if task == "cmnli":
                    return {
                        "蕴含": "0",
                        "中立": "1",
                        "矛盾": "2",
                    }[canonical]

    return "__invalid__"


In [6]:

from mlx_lm import load as mlx_load
from mlx_lm import generate as mlx_generate

def load_transformers_model(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )

    model.eval()
    return tokenizer, model

def generate_tf(tokenizer, model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=6,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

def evaluate_model(model_key, model_cfg):
    model_id = model_cfg["model_id"]
    backend = model_cfg["backend"]

    print(f"Loading {model_id}")

    if backend == "transformers":
        tokenizer, model = load_transformers_model(model_id)
    else:
        model, tokenizer = mlx_load(model_id)

    summaries = []

    for task in TASKS:
        dataset = load_dataset("clue", task, split="validation")
        dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))

        tnews_mapping = None

        if task == "tnews":
            label_names = dataset.features["label"].names
            tnews_mapping = {i: v for i, v in enumerate(label_names)}

        y_true = []
        y_pred = []

        start = time.time()

        for ex in tqdm(dataset, desc=f"{model_key}/{task}"):
            prompt = build_prompt(task, ex)

            if backend == "transformers":
                raw = generate_tf(tokenizer, model, prompt)
            else:
                raw = mlx_generate(model, tokenizer, prompt=prompt, max_tokens=6)

            pred = extract_label(task, raw, tnews_mapping)

            y_true.append(str(ex["label"]))
            y_pred.append(pred)

        elapsed = time.time() - start

        valid_preds = [p if p != "__invalid__" else "-1" for p in y_pred]

        summaries.append({
            "model_key": model_key,
            "model_id": model_id,
            "task": task,
            "accuracy": accuracy_score(y_true, valid_preds),
            "macro_f1": f1_score(y_true, valid_preds, average="macro"),
            "invalid_rate": sum(p == "__invalid__" for p in y_pred) / len(y_pred),
            "seconds": elapsed,
        })

    if backend == "transformers":
        del model
        del tokenizer

    cleanup()

    return summaries


In [7]:

all_results = []

for model_key, model_cfg in MODELS.items():
    results = evaluate_model(model_key, model_cfg)
    all_results.extend(results)

results_df = pd.DataFrame(all_results)
results_df


Loading Qwen/Qwen3-4B-Instruct-2507


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

qwen3_4b_instruct_2507/afqmc:  10%|█▏          | 19/200 [00:13<02:08,  1.40it/s]


KeyboardInterrupt: 